# Hourly Scanner

**Run:** During market hours, after 10:30 AM ET / 8:00 PM IST

**Scans:** Option Sell (Chirag Rathod) · Rathod Intraday Bullish · NR7 · Momentum · Dual BB 15m/1h · **BullBhai Bullish Momentum** · **Short Term Breakout** · **Potential Breakouts** · **100% Buy Breakout** · **Possible Bottom Out (Weekly)**

**Universes:** S&P 500 · Russell 2000 · Nifty 500 (toggle with `INCLUDE_NIFTY`)

**Re-run any time** during the trading day to refresh signals

**Publishes to:** https://docs.google.com/spreadsheets/d/1rzc_6fZoHMFi1Ee75zRmIuGxeCWog1E62pZp9c1Lsfs

**Runtime > Run all** — takes ~15-20 min


In [ ]:
# CELL 1 — Install
import subprocess, sys
subprocess.check_call([sys.executable,"-m","pip","install","-q",
    "yfinance","tqdm","requests","lxml","gspread","gspread-dataframe"])
print("Ready")

Ready


In [ ]:
# @title Default title text {"display-mode":"form"}
# CELL 2 — Hourly scanner settings
import warnings; warnings.filterwarnings("ignore")
import requests, io, time, gc
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime

# ══════════════════════════════════════════════════════════════
#  EDIT THESE DIRECTLY — set to True or False, then run this cell
# ══════════════════════════════════════════════════════════════

# ── Universe ────────────────────────────────────────────────
INCLUDE_NIFTY  = False    # False = skip Nifty 500 entirely
INCLUDE_RUSSEL = False

# ── Existing scan toggles ────────────────────────────────────
RUN_OPTION_SELL    = True    # Chirag Rathod bearish 1h reversal
RUN_RATHOD_BULLISH = True    # Rathod daily+1h uptrend
RUN_NR7            = True    # Narrow Range 7-day breakout
RUN_MOMENTUM       = True    # Momentum scan
RUN_DUALBB_15M     = True    # Dual BB crossover — 15 min
RUN_DUALBB_1H      = True    # Dual BB crossover — 1 hour

# ── New Chartink scans ───────────────────────────────────────
RUN_BULLBHAI       = True    # BullBhai Bullish Momentum (15m close > EMA200, daily close > 2d high, RSI>51)
RUN_STB            = True    # Short Term Breakout (5d close > 120d high*1.05, vol>SMA5, close>prior)
RUN_POT_BREAKOUT   = True    # Potential Breakout (close*1.05 > 200d high, 30d high <= 8d high 30d ago, vol>SMA50, close>90)
RUN_BUY_BREAKOUT   = True    # 100% Buy Breakout at open (widest range 8 days, SMA20>40>60, vol surge 1.25x)
RUN_BOTTOM_OUT     = True    # Possible Bottom Out Weekly (4-week descending highs/lows then bullish reversal)

# ══════════════════════════════════════════════════════════════

BATCH_SIZE            = 40
SLEEP_BETWEEN_BATCHES = 3
SHEET_ID = "1rzc_6fZoHMFi1Ee75zRmIuGxeCWog1E62pZp9c1Lsfs"

print(f"Nifty included     : {INCLUDE_NIFTY}")
print(f"Russel included    : {INCLUDE_RUSSEL}")
print()
print("Scans enabled:")
print(f"  Option Sell       : {RUN_OPTION_SELL}")
print(f"  Rathod Bullish    : {RUN_RATHOD_BULLISH}")
print(f"  NR7               : {RUN_NR7}")
print(f"  Momentum          : {RUN_MOMENTUM}")
print(f"  Dual BB 15min     : {RUN_DUALBB_15M}")
print(f"  Dual BB 1h        : {RUN_DUALBB_1H}")
print(f"  BullBhai Momentum : {RUN_BULLBHAI}")
print(f"  Short Term BO     : {RUN_STB}")
print(f"  Potential BO      : {RUN_POT_BREAKOUT}")
print(f"  100% Buy BO       : {RUN_BUY_BREAKOUT}")
print(f"  Bottom Out Weekly : {RUN_BOTTOM_OUT}")


In [ ]:
# CELL 3 — Load tickers
HEADERS = {"User-Agent":"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"}

def get_sp500():
    try:
        html = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",headers=HEADERS,timeout=15).text
        t = [str(x).replace(".","-") for x in pd.read_html(io.StringIO(html))[0]["Symbol"].tolist()]
        print(f"  S&P 500      : {len(t)}"); return t
    except Exception as e:
        print(f"  SP500 failed: {e}"); return []

def get_russell2000():
    try:
        nyse = requests.get("https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/main/nyse/nyse_tickers.txt",headers=HEADERS,timeout=15).text.strip().split()
        nasd = requests.get("https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/main/nasdaq/nasdaq_tickers.txt",headers=HEADERS,timeout=15).text.strip().split()
        sp_csv = requests.get("https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv",headers=HEADERS,timeout=15).text
        sp_set = set(pd.read_csv(io.StringIO(sp_csv))["Symbol"].str.replace(".","-",regex=False).tolist())
        all_t = list(dict.fromkeys(nyse+nasd))
        t = [x for x in all_t if x.isalpha() and 2<=len(x)<=4 and x not in sp_set][:2000]
        print(f"  Russell 2000 : {len(t)}"); return t
    except Exception as e:
        print(f"  R2K failed: {e}"); return []

def get_nifty500():
    symbols = [
        "360ONE","3MINDIA","ABB","ACC","ACMESOLAR","AIAENG","APLAPOLLO","AUBANK",
        "AWL","AADHARHFC","AARTIIND","AAVAS","ABBOTINDIA","ACE","ACUTAAS","ADANIENSOL",
        "ADANIENT","ADANIGREEN","ADANIPORTS","ADANIPOWER","ATGL","ABCAPITAL","ABFRL","ABLBL",
        "ABREL","ABSLAMC","CPPLUS","AEGISLOG","AEGISVOPAK","AFCONS","AFFLE","AJANTPHARM",
        "ALKEM","ABDL","AMBER","AMBUJACEM","ANANDRATHI","ANANTRAJ","ANGELONE","ANTHEM",
        "ANURAS","APARINDS","APOLLOHOSP","APOLLOTYRE","APTUS","ASAHIINDIA","ASHOKLEY","ASIANPAINT",
        "ASTERDM","ASTRAL","ATHERENERG","ATUL","AUROPHARMA","AIIL","DMART","AXISBANK",
        "BEML","BLS","BSE","BAJAJ-AUTO","BAJFINANCE","BAJAJFINSV","BAJAJHLDNG","BAJAJHFL",
        "BALKRISIND","BALRAMCHIN","BANDHANBNK","BANKBARODA","BANKINDIA","MAHABANK","BATAINDIA","BAYERCROP",
        "BELRISE","BERGEPAINT","BDL","BEL","BHARATFORG","BHEL","BPCL","BHARTIARTL",
        "BHARTIHEXA","BIKAJI","GROWW","BIOCON","BSOFT","BLUEDART","BLUEJET","BLUESTARCO",
        "BBTC","BOSCHLTD","FIRSTCRY","BRIGADE","BRITANNIA","MAPMYINDIA","CCL","CESC",
        "CGPOWER","CIEINDIA","CRISIL","CANFINHOME","CANBK","CANHLIFE","CAPLIPOINT","CGCL",
        "CARBORUNIV","CARTRADE","CASTROLIND","CEATLTD","CEMPRO","CENTRALBK","CDSL","CHALET",
        "CHAMBLFERT","CHENNPETRO","CHOICEIN","CHOLAHLDNG","CHOLAFIN","CIPLA","CUB","CLEAN",
        "COALINDIA","COCHINSHIP","COFORGE","COHANCE","COLPAL","CAMS","CONCORDBIO","CONCOR",
        "COROMANDEL","CRAFTSMAN","CREDITACC","CROMPTON","CUMMINSIND","CYIENT","DCMSHRIRAM","DLF",
        "DOMS","DABUR","DALBHARAT","DATAPATTNS","DEEPAKFERT","DEEPAKNTR","DELHIVERY","DEVYANI",
        "DIVISLAB","DIXON","LALPATHLAB","DRREDDY","EIDPARRY","EIHOTEL","EICHERMOT","ELECON",
        "ELGIEQUIP","EMAMILTD","EMCURE","EMMVEE","ENDURANCE","ENGINERSIN","ERIS","ESCORTS",
        "ETERNAL","EXIDEIND","NYKAA","FEDERALBNK","FACT","FINCABLES","FSL","FIVESTAR",
        "FORCEMOT","FORTIS","GAIL","GMRAIRPORT","GABRIEL","GALLANTT","GRSE","GICRE",
        "GILLETTE","GLAND","GLAXO","GLENMARK","MEDANTA","GODIGIT","GPIL","GODFRYPHLP",
        "GODREJCP","GODREJIND","GODREJPROP","GRANULES","GRAPHITE","GRASIM","GRAVITA","GESHIP",
        "FLUOROCHEM","GMDCLTD","HEG","HBLENGINE","HCLTECH","HDBFS","HDFCAMC","HDFCBANK",
        "HDFCLIFE","HFCL","HAVELLS","HEROMOTOCO","HEXT","HSCL","HINDALCO","HAL",
        "HINDCOPPER","HINDPETRO","HINDUNILVR","HINDZINC","POWERINDIA","HOMEFIRST","HONASA","HONAUT",
        "HUDCO","HYUNDAI","ICICIBANK","ICICIGI","ICICIAMC","ICICIPRULI","IDBI","IDFCFIRSTB",
        "IFCI","IIFL","IRB","IRCON","ITCHOTELS","ITC","ITI","INDGN",
        "INDIACEM","INDIAMART","INDIANB","IEX","INDHOTEL","IOC","IOB","IRCTC",
        "IRFC","IREDA","IGL","INDUSTOWER","INDUSINDBK","NAUKRI","INFY","INOXWIND",
        "INTELLECT","INDIGO","IGIL","IKS","IPCALAB","JBCHEPHARM","JKCEMENT","JBMA",
        "JKTYRE","JMFINANCIL","JSWCEMENT","JSWDULUX","JSWENERGY","JSWINFRA","JSWSTEEL","JAINREC",
        "JPPOWER","JINDALSAW","JSL","JINDALSTEL","JIOFIN","JUBLFOOD","JUBLINGREA","JUBLPHARMA",
        "JWL","JYOTICNC","KPRMILL","KEI","KPITTECH","KAJARIACER","KPIL","KALYANKJIL",
        "KARURVYSYA","KAYNES","KEC","KFINTECH","KIRLOSENG","KOTAKBANK","KIMS","LTF",
        "LTTS","LGEINDIA","LICHSGFIN","LTFOODS","LTM","LT","LATENTVIEW","LAURUSLABS",
        "THELEELA","LEMONTREE","LENSKART","LICI","LINDEINDIA","LLOYDSME","LODHA","LUPIN",
        "MMTC","MRF","MGL","M&MFIN","M&M","MANAPPURAM","MRPL","MANKIND",
        "MARICO","MARUTI","MFSL","MAXHEALTH","MAZDOCK","MEESHO","MINDACORP","MSUMI",
        "MOTILALOFS","MPHASIS","MCX","MUTHOOTFIN","NATCOPHARM","NBCC","NCC","NHPC",
        "NLCINDIA","NMDC","NSLNISP","NTPCGREEN","NTPC","NH","NATIONALUM","NAVA",
        "NAVINFLUOR","NESTLEIND","NETWEB","NEULANDLAB","NEWGEN","NAM-INDIA","NIVABUPA","NUVAMA",
        "NUVOCO","OBEROIRLTY","ONGC","OIL","OLAELEC","OLECTRA","PAYTM","ONESOURCE",
        "OFSS","POLICYBZR","PCBL","PGEL","PIIND","PNBHOUSING","PTCIL","PVRINOX",
        "PAGEIND","PARADEEP","PATANJALI","PERSISTENT","PETRONET","PFIZER","PHOENIXLTD","PWL",
        "PIDILITIND","PINELABS","PIRAMALFIN","PPLPHARMA","POLYMED","POLYCAB","POONAWALLA","PFC",
        "POWERGRID","PREMIERENE","PRESTIGE","PNB","RRKABEL","RBLBANK","RECLTD","RHIM",
        "RITES","RADICO","RVNL","RAILTEL","RAINBOW","RKFORGE","REDINGTON","RELIANCE",
        "RPOWER","SBFC","SBICARD","SBILIFE","SJVN","SRF","SAGILITY","SAILIFE",
        "SAMMAANCAP","MOTHERSON","SAPPHIRE","SARDAEN","SAREGAMA","SCHAEFFLER","SCHNEIDER","SCI",
        "SHREECEM","SHRIRAMFIN","SHYAMMETL","ENRIN","SIEMENS","SIGNATURE","SOBHA","SOLARINDS",
        "SONACOMS","SONATSOFTW","STARHEALTH","SBIN","SAIL","SUMICHEM","SUNPHARMA","SUNTV",
        "SUNDARMFIN","SUPREMEIND","SPLPETRO","SUZLON","SWANCORP","SWIGGY","SYNGENE","SYRMA",
        "TBOTEK","TVSMOTOR","TATACAP","TATACHEM","TATACOMM","TCS","TATACONSUM","TATAELXSI",
        "TATAINVEST","TMCV","TMPV","TATAPOWER","TATASTEEL","TATATECH","TTML","TECHM",
        "TECHNOE","TEGA","TEJASNET","TENNIND","NIACL","RAMCOCEM","THERMAX","TIMKEN",
        "TITAGARH","TITAN","TORNTPHARM","TORNTPOWER","TARIL","TRAVELFOOD","TRENT","TRIDENT",
        "TRITURBINE","TIINDIA","UCOBANK","UNOMINDA","UPL","UTIAMC","ULTRACEMCO","UNIONBANK",
        "UBL","UNITDSPR","URBANCO","USHAMART","VTL","VBL","VEDL","VIJAYA",
    ]
    t = [s+".NS" for s in list(dict.fromkeys(symbols))]
    print(f"  Nifty 500    : {len(t)}"); return t

print("Loading tickers...")
sp500      = get_sp500()
r2000      = get_russell2000() if INCLUDE_RUSSEL else []
r2000_only = [t for t in r2000 if t not in set(sp500)]
nifty500   = get_nifty500() if INCLUDE_NIFTY else []
print(f"  Total        : {len(sp500)+len(r2000_only)+len(nifty500)}")

Loading tickers...
  S&P 500      : 503
  Total        : 503


In [ ]:
# CELL 4 — Indicators
import pandas as pd
import numpy as np

def sma(s,n): return s.rolling(n).mean()
def ema(s,n): return s.ewm(span=n,adjust=False).mean()
def rsi(s,n=14):
    d=s.diff()
    g=d.clip(lower=0).rolling(n).mean()
    l=(-d.clip(upper=0)).rolling(n).mean()
    return 100-(100/(1+g/l.replace(0,np.nan)))
def _f(x):
    try: return float(x)
    except: return float("nan")
def _range(d,i):
    try: return _f(d["High"].iloc[-(i+1)]) - _f(d["Low"].iloc[-(i+1)])
    except: return float("nan")
def vwap_intraday(h):
    h=h.copy()
    h["_d"]=h.index.normalize()
    h["_tp"]=(h["High"]+h["Low"]+h["Close"])/3
    tpv=h.groupby("_d").apply(lambda g:(g["_tp"]*g["Volume"]).cumsum()).values
    cvol=h.groupby("_d")["Volume"].cumsum().values
    return pd.Series(tpv/cvol,index=h.index)
print("Indicators ready")

Indicators ready


In [ ]:
# CELL 5 — Fetchers (daily/weekly + intraday separated)
import yfinance as yf
import pandas as pd
import time

def fetch_batch(tickers, period, interval, retries=2):
    """Daily / weekly batch fetcher."""
    out = {}
    if not tickers: return out
    for attempt in range(retries+1):
        try:
            raw = yf.download(tickers, period=period, interval=interval,
                              group_by="ticker", auto_adjust=False,
                              progress=False, threads=True)
            if raw.empty: break
            if len(tickers)==1:
                t=tickers[0]; df=raw.copy()
                if isinstance(df.columns,pd.MultiIndex): df.columns=df.columns.get_level_values(1)
                df=df.drop(columns=["Adj Close"],errors="ignore")
                df.dropna(how="all",inplace=True)
                if len(df)>5: out[t]=df
            else:
                for t in tickers:
                    try:
                        if t in raw.columns.levels[0]:
                            df=raw[t].copy()
                            if isinstance(df.columns,pd.MultiIndex): df.columns=df.columns.get_level_values(-1)
                            df=df.drop(columns=["Adj Close"],errors="ignore")
                            df=df.dropna(how="all")
                            if len(df)>5: out[t]=df
                    except: pass
            break
        except Exception as e:
            if any(x in str(e) for x in ["Rate","429","Too Many","RateLimit"]):
                wait=30*(attempt+1); print(f"  Rate limit — wait {wait}s"); time.sleep(wait)
            else: break
    return out

def fetch_intraday(tickers, period="5d", interval="1h", retries=2):
    """Intraday fetcher (1h) — uses your working MultiIndex fix."""
    out = {}
    if not tickers: return out
    for attempt in range(retries+1):
        try:
            raw = yf.download(tickers, period=period, interval=interval,
                              group_by="ticker", auto_adjust=False,
                              progress=False, threads=True)
            if raw.empty: break
            if len(tickers)==1:
                t=tickers[0]; df=raw.copy()
                if isinstance(df.columns,pd.MultiIndex): df.columns=df.columns.get_level_values(1)
                df=df.drop(columns=["Adj Close"],errors="ignore")
                df.dropna(how="all",inplace=True)
                if len(df)>5: out[t]=df
            else:
                for t in tickers:
                    try:
                        if t in raw.columns.levels[0]:
                            df=raw[t].copy()
                            if isinstance(df.columns,pd.MultiIndex): df.columns=df.columns.get_level_values(-1)
                            df=df.drop(columns=["Adj Close"],errors="ignore")
                            df=df.dropna(how="all")
                            if len(df)>5: out[t]=df
                    except: pass
            break
        except Exception as e:
            if any(x in str(e) for x in ["Rate","429","Too Many","RateLimit"]):
                wait=30*(attempt+1); print(f"  Rate limit — wait {wait}s"); time.sleep(wait)
            else: break
    return out

print("fetch_batch (daily/weekly) + fetch_intraday (1h) ready")

fetch_batch (daily/weekly) + fetch_intraday (1h) ready


In [ ]:
# CELL 6 — Hourly scan functions
import numpy as np

def scan_option_sell(d, h):
    """Chirag Rathod: Bearish 1h reversal at VWAP, daily below SMA20."""
    try:
        if len(h)<4 or len(d)<21: return False
        h=h.copy(); h["vwap"]=vwap_intraday(h)
        b1=h.iloc[-2]; b2=h.iloc[-3]; v1=_f(h["vwap"].iloc[-2])
        d=d.copy(); d["s20"]=sma(d["Close"],20)
        return (b1["Close"]<b1["Open"] and b1["Open"]<b1["High"]
            and b1["Close"]>b1["Low"] and b1["Close"]<b1["High"]
            and b1["Close"]<v1 and b1["Open"]>v1 and b1["High"]>v1
            and b2["Close"]>b2["Open"]
            and b1["Open"]>b2["Open"] and b1["Open"]>b2["Close"]
            and _f(d["Close"].iloc[-1])<_f(d["s20"].iloc[-1]))
    except: return False

def scan_rathod_bullish(d, h):
    """Rathod: Daily+1h uptrend — above SMA20, 5 rising lows, RSI>40."""
    try:
        if len(d)<22 or len(h)<8: return False
        d=d.copy(); h=h.copy()
        d["s20"]=sma(d["Close"],20); h["s20"]=sma(h["Close"],20)
        dc=_f(d["Close"].iloc[-1]); hc=_f(h["Close"].iloc[-1])
        return (dc>_f(d["s20"].iloc[-1])
            and all(dc>_f(d["Low"].iloc[-i]) for i in range(2,7))
            and _f(rsi(d["Close"]).iloc[-1])>40
            and hc>_f(h["s20"].iloc[-1])
            and all(hc>_f(h["Low"].iloc[-i]) for i in range(2,7))
            and _f(rsi(h["Close"]).iloc[-1])>40)
    except: return False

def scan_nr7(d):
    """NR7: Today's daily range (high-low) is the narrowest of the last 7 days.
    TradingView: daily(high-low) < N days ago(high-low) for N=1..7
    Also requires: close>open, close>yesterday close, weekly close>weekly open,
    monthly close>monthly open, yesterday volume>10000,
    SMA20>SMA40>SMA60 (trend alignment), today volume > yesterday volume * 1.25"""
    try:
        if len(d) < 9: return False
        r0 = _range(d, 0)  # today's range
        # Range must be narrower than each of the last 7 days
        range_check = all(r0 < _range(d, i) for i in range(1, 8))
        if not range_check: return False

        close = _f(d["Close"].iloc[-1])
        open_ = _f(d["Open"].iloc[-1])
        prev_close = _f(d["Close"].iloc[-2])
        prev_vol   = _f(d["Volume"].iloc[-2])
        today_vol  = _f(d["Volume"].iloc[-1])

        if not (close > open_): return False
        if not (close > prev_close): return False
        if not (prev_vol > 10000): return False
        if not (today_vol > prev_vol * 1.25): return False

        # SMA trend alignment: SMA20 > SMA40 > SMA60
        s20 = _f(sma(d["Close"], 20).iloc[-1])
        s40 = _f(sma(d["Close"], 40).iloc[-1])
        s60 = _f(sma(d["Close"], 60).iloc[-1])
        if not (s20 > s40 > s60): return False

        return True
    except: return False

def scan_nr7_weekly_monthly(d, w, m):
    """Same as NR7 but also checks weekly close>open and monthly close>open.
    Pass d=daily, w=weekly, m=monthly dataframes."""
    try:
        if not scan_nr7(d): return False
        if w is None or len(w) < 2: return False
        if m is None or len(m) < 2: return False
        wk_close = _f(w["Close"].iloc[-1]); wk_open = _f(w["Open"].iloc[-1])
        mo_close = _f(m["Close"].iloc[-1]); mo_open = _f(m["Open"].iloc[-1])
        return (wk_close > wk_open) and (mo_close > mo_open)
    except: return False

def scan_momentum(d):
    """Momentum: close>50, vol>avgvol20, SMA20>SMA50, RSI>50,
    close>SMA50, close>yesterday high, avgvol20>500k"""
    try:
        if len(d) < 52: return False
        close = _f(d["Close"].iloc[-1])
        vol   = _f(d["Volume"].iloc[-1])
        avg_vol_20 = _f(sma(d["Volume"], 20).iloc[-1])
        s20 = _f(sma(d["Close"], 20).iloc[-1])
        s50 = _f(sma(d["Close"], 50).iloc[-1])
        r14 = _f(rsi(d["Close"], 14).iloc[-1])
        prev_high = _f(d["High"].iloc[-2])

        return (
            close > 50
            and vol > avg_vol_20
            and s20 > s50
            and r14 > 50
            and close > s50
            and close > prev_high
            and avg_vol_20 > 500000
        )
    except: return False

def scan_dualbb_15m(d_intraday, daily_rsi_val):
    """Dual BB Screener — 15min timeframe.
    BB1 (fast, len=5, mult=0.2611) lower crosses above BB2 (slow, len=20, mult=0.2611) lower
       AND price > 200 SMA AND daily RSI > 50  -> BULLISH
    BB1 upper crosses below BB2 upper
       AND price < 200 SMA AND daily RSI < 50  -> BEARISH
    Returns "bullish", "bearish", or None.
    d_intraday: 15-minute OHLC dataframe (needs 200+ bars for SMA200)
    daily_rsi_val: daily RSI(14) value (computed separately, passed in)
    """
    try:
        if len(d_intraday) < 201: return None
        c = d_intraday["Close"]

        bb1_basis = sma(c, 5);  bb1_dev = 0.2611 * c.rolling(5).std()
        bb1_upper = bb1_basis + bb1_dev; bb1_lower = bb1_basis - bb1_dev

        bb2_basis = sma(c, 20); bb2_dev = 0.2611 * c.rolling(20).std()
        bb2_upper = bb2_basis + bb2_dev; bb2_lower = bb2_basis - bb2_dev

        sma200 = sma(c, 200)

        # Crossover: bb1_lower crosses above bb2_lower (today vs yesterday bar)
        buy_cross = (bb1_lower.iloc[-1] > bb2_lower.iloc[-1]) and (bb1_lower.iloc[-2] <= bb2_lower.iloc[-2])
        # Crossunder: bb1_upper crosses below bb2_upper
        sell_cross = (bb1_upper.iloc[-1] < bb2_upper.iloc[-1]) and (bb1_upper.iloc[-2] >= bb2_upper.iloc[-2])

        close = _f(c.iloc[-1])
        above_sma200 = close > _f(sma200.iloc[-1])
        below_sma200 = close < _f(sma200.iloc[-1])

        if buy_cross and above_sma200 and daily_rsi_val > 50:
            return "bullish"
        if sell_cross and below_sma200 and daily_rsi_val < 50:
            return "bearish"
        return None
    except: return None

def scan_dualbb_1h(d_hourly, daily_rsi_val):
    """Dual BB Screener — 1 hour timeframe. Same logic as 15min, applied to hourly bars."""
    try:
        if len(d_hourly) < 201: return None
        c = d_hourly["Close"]

        bb1_basis = sma(c, 5);  bb1_dev = 0.2611 * c.rolling(5).std()
        bb1_upper = bb1_basis + bb1_dev; bb1_lower = bb1_basis - bb1_dev

        bb2_basis = sma(c, 20); bb2_dev = 0.2611 * c.rolling(20).std()
        bb2_upper = bb2_basis + bb2_dev; bb2_lower = bb2_basis - bb2_dev

        sma200 = sma(c, 200)

        buy_cross = (bb1_lower.iloc[-1] > bb2_lower.iloc[-1]) and (bb1_lower.iloc[-2] <= bb2_lower.iloc[-2])
        sell_cross = (bb1_upper.iloc[-1] < bb2_upper.iloc[-1]) and (bb1_upper.iloc[-2] >= bb2_upper.iloc[-2])

        close = _f(c.iloc[-1])
        above_sma200 = close > _f(sma200.iloc[-1])
        below_sma200 = close < _f(sma200.iloc[-1])

        if buy_cross and above_sma200 and daily_rsi_val > 50:
            return "bullish"
        if sell_cross and below_sma200 and daily_rsi_val < 50:
            return "bearish"
        return None
    except: return None

print("Hourly scan functions ready:")
print("  option_sell, rathod_bullish, nr7, momentum, dualbb_15m, dualbb_1h")

# ── NEW CHARTINK SCANS ─────────────────────────────────────────────────────

def scan_bullbhai_momentum(d, m15):
    """BullBhai Bullish Momentum (Chartink):
    - 15min close > EMA(close, 200) on 15min chart
    - daily close > 2 days ago high
    - daily RSI(14) > 51
    d    : daily OHLC dataframe
    m15  : 15-minute OHLC dataframe (needs 200+ bars)
    """
    try:
        if len(d) < 5 or len(m15) < 201: return False
        # 15min: close > EMA(close, 200)
        ema200_15m = _f(ema(m15["Close"], 200).iloc[-1])
        close_15m  = _f(m15["Close"].iloc[-1])
        if not (close_15m > ema200_15m): return False
        # Daily: close > 2 days ago high
        daily_close    = _f(d["Close"].iloc[-1])
        two_days_high  = _f(d["High"].iloc[-3])   # [-1]=today, [-2]=1d ago, [-3]=2d ago
        if not (daily_close > two_days_high): return False
        # Daily RSI(14) > 51
        daily_rsi_val = _f(rsi(d["Close"], 14).iloc[-1])
        return daily_rsi_val > 51
    except: return False


def scan_short_term_breakout(d):
    """Short Term Breakout (Chartink):
    - max(5, daily close) > 6 days ago max(120, daily close) * 1.05
      i.e. highest close in last 5 days > (highest close in 120 days, 6 bars ago) * 1.05
    - daily volume > SMA(volume, 5)
    - daily close > 1 day ago close
    d : daily OHLC (needs 130+ bars)
    """
    try:
        if len(d) < 130: return False
        # max(5, daily close): rolling max of close over 5 bars up to today
        max5_close = _f(d["Close"].rolling(5).max().iloc[-1])
        # 6 days ago max(120, daily close): rolling 120-bar close max, taken 6 bars ago
        max120_6ago = _f(d["Close"].rolling(120).max().iloc[-7])  # -7 = 6 days ago
        if not (max5_close > max120_6ago * 1.05): return False
        # Volume > SMA(volume, 5)
        vol_today = _f(d["Volume"].iloc[-1])
        vol_sma5  = _f(sma(d["Volume"], 5).iloc[-1])
        if not (vol_today > vol_sma5): return False
        # Close > 1 day ago close
        return _f(d["Close"].iloc[-1]) > _f(d["Close"].iloc[-2])
    except: return False


def scan_potential_breakout(d):
    """Potential Breakouts (Chartink):
    - daily close * 1.05 > max(200, daily high)       → within 5% of 200-day high
    - max(30, daily high) <= 30 days ago max(8, daily high)  → range contracting
    - daily volume > SMA(daily volume, 50)
    - daily close > 90
    d : daily OHLC (needs 230+ bars)
    """
    try:
        if len(d) < 230: return False
        close_today = _f(d["Close"].iloc[-1])
        # close * 1.05 > 200d high  → price within 5% of 200d high
        high200 = _f(d["High"].rolling(200).max().iloc[-1])
        if not (close_today * 1.05 > high200): return False
        # max(30, daily high) <= 30 days ago max(8, daily high)
        # i.e. 30-day high range is NOT expanding beyond where 8-day high was 30 days ago
        high30_now      = _f(d["High"].rolling(30).max().iloc[-1])
        high8_30ago     = _f(d["High"].rolling(8).max().iloc[-31])  # 8-day max taken 30 bars ago
        if not (high30_now <= high8_30ago): return False
        # Volume > SMA(volume, 50)
        vol_sma50 = _f(sma(d["Volume"], 50).iloc[-1])
        if not (_f(d["Volume"].iloc[-1]) > vol_sma50): return False
        # Close > 90
        return close_today > 90
    except: return False


def scan_buy_breakout_100(d):
    """100% Buy Breakout at 9:30am (Chartink):
    Today's range (high-low) is WIDEST of last 8 days AND:
    - daily close > daily open (bullish candle)
    - daily close > 1 day ago close
    - weekly close > weekly open  (approximated: this week's close > Monday open)
    - monthly close > monthly open (approximated: this month's close > month open)
    - 1 day ago volume > 10,000
    - SMA(close,20) > SMA(close,40) > SMA(close,60)
    - today volume > 1 day ago volume * 1.25
    d : daily OHLC (needs 65+ bars for SMA60)
    Note: weekly/monthly alignment approximated from daily data.
    """
    try:
        if len(d) < 65: return False
        # Today's range is widest of last 8 days (today + 7 prior)
        today_range = _f(d["High"].iloc[-1]) - _f(d["Low"].iloc[-1])
        for i in range(1, 8):
            if not (today_range > _range(d, i)): return False
        # Bullish candle
        if not (_f(d["Close"].iloc[-1]) > _f(d["Open"].iloc[-1])): return False
        # Close > prior close
        if not (_f(d["Close"].iloc[-1]) > _f(d["Close"].iloc[-2])): return False
        # Prior day volume > 10,000
        if not (_f(d["Volume"].iloc[-2]) > 10000): return False
        # Today volume > prior volume * 1.25
        if not (_f(d["Volume"].iloc[-1]) > _f(d["Volume"].iloc[-2]) * 1.25): return False
        # SMA trend alignment
        s20 = _f(sma(d["Close"], 20).iloc[-1])
        s40 = _f(sma(d["Close"], 40).iloc[-1])
        s60 = _f(sma(d["Close"], 60).iloc[-1])
        if not (s20 > s40 > s60): return False
        # Weekly & monthly approximation from daily data:
        # Weekly: last 5 days — if most-recent Mon open exists, use it; else use 5d ago open
        week_open  = _f(d["Open"].iloc[-5])   # approx start of week
        week_close = _f(d["Close"].iloc[-1])
        if not (week_close > week_open): return False
        # Monthly: use first trading day of month vs today
        month_open  = _f(d["Open"].iloc[-22])  # approx ~1 month ago
        month_close = _f(d["Close"].iloc[-1])
        return month_close > month_open
    except: return False


def scan_possible_bottom_out(d, w):
    """Possible Bottom Out Weekly (Chartink):
    Weekly pattern — 3 descending bearish candles followed by a bullish reversal:
    - 4w ago high > 3w ago high  AND  4w ago low > 3w ago low  AND  4w ago close < 4w ago open
    - 3w ago high > 2w ago high  AND  3w ago low > 2w ago low  AND  3w ago close < 3w ago open
    - 2w ago high > 1w ago high  AND  2w ago low > 1w ago low  AND  2w ago close < 2w ago open
    - this week high > 1w ago high  AND  this week low > 1w ago low  AND  weekly close > weekly open
    - weekly close >= 100
    w : weekly OHLC dataframe (needs 6+ bars)
    """
    try:
        if w is None or len(w) < 6: return False
        # Index from end: -1=this week, -2=1w ago, -3=2w ago, -4=3w ago, -5=4w ago
        def wh(i): return _f(w["High"].iloc[i])
        def wl(i): return _f(w["Low"].iloc[i])
        def wc(i): return _f(w["Close"].iloc[i])
        def wo(i): return _f(w["Open"].iloc[i])
        # 4w ago descending bearish
        if not (wh(-5) > wh(-4) and wl(-5) > wl(-4) and wc(-5) < wo(-5)): return False
        # 3w ago descending bearish
        if not (wh(-4) > wh(-3) and wl(-4) > wl(-3) and wc(-4) < wo(-4)): return False
        # 2w ago descending bearish
        if not (wh(-3) > wh(-2) and wl(-3) > wl(-2) and wc(-3) < wo(-3)): return False
        # This week: bullish reversal — higher high, higher low, close > open
        if not (wh(-1) > wh(-2) and wl(-1) > wl(-2) and wc(-1) > wo(-1)): return False
        # Weekly close >= 100
        return wc(-1) >= 100
    except: return False

print("New Chartink scan functions ready:")
print("  bullbhai_momentum, short_term_breakout, potential_breakout")
print("  buy_breakout_100, possible_bottom_out")


In [ ]:
# CELL 7 — Hourly scan runner (respects scan toggles from Cell 2)
import time

# Build the active scan key list based on toggles
HOURLY_SCAN_KEYS = []
if RUN_OPTION_SELL:    HOURLY_SCAN_KEYS.append("option_sell")
if RUN_RATHOD_BULLISH: HOURLY_SCAN_KEYS.append("rathod_bullish")
if RUN_NR7:            HOURLY_SCAN_KEYS.append("nr7")
if RUN_MOMENTUM:       HOURLY_SCAN_KEYS.append("momentum")
if RUN_DUALBB_15M:     HOURLY_SCAN_KEYS += ["dualbb_15m_bull", "dualbb_15m_bear"]
if RUN_DUALBB_1H:      HOURLY_SCAN_KEYS += ["dualbb_1h_bull", "dualbb_1h_bear"]
if RUN_BULLBHAI:       HOURLY_SCAN_KEYS.append("bullbhai_momentum")
if RUN_STB:            HOURLY_SCAN_KEYS.append("short_term_breakout")
if RUN_POT_BREAKOUT:   HOURLY_SCAN_KEYS.append("potential_breakout")
if RUN_BUY_BREAKOUT:   HOURLY_SCAN_KEYS.append("buy_breakout_100")
if RUN_BOTTOM_OUT:     HOURLY_SCAN_KEYS.append("possible_bottom_out")

print(f"Active scans: {HOURLY_SCAN_KEYS}")
if not HOURLY_SCAN_KEYS:
    print("WARNING: No scans enabled! Check Cell 2 toggles.")

def run_hourly_scans(tickers, label):
    is_nifty = label == "Nifty 500"
    hits = {k: [] for k in HOURLY_SCAN_KEYS}
    if not HOURLY_SCAN_KEYS:
        return hits

    batches = [tickers[i:i+BATCH_SIZE] for i in range(0, len(tickers), BATCH_SIZE)]
    total = len(batches)
    print(f"Scanning {label} — {len(tickers)} tickers, {total} batches")

    # Determine what data to fetch
    need_hourly    = RUN_OPTION_SELL or RUN_RATHOD_BULLISH
    need_15m       = RUN_DUALBB_15M or RUN_BULLBHAI   # BullBhai needs 15m for EMA200
    need_15m_long  = RUN_BULLBHAI                      # BullBhai needs 200+ 15m bars (~1mo)
    need_1h_long   = RUN_DUALBB_1H
    need_weekly    = RUN_NR7 or RUN_BOTTOM_OUT         # Bottom Out needs weekly data
    # STB needs 130d, PotBO needs 230d, BuyBO needs 65d — all covered by 1y daily
    need_daily_2y  = RUN_STB or RUN_POT_BREAKOUT       # need longer history

    for n, batch in enumerate(batches):
        print(f"  Batch {n+1}/{total}", end="\r")

        # Daily: 2y for STB/PotBO, else 1y
        daily_period = "2y" if need_daily_2y else "1y"
        daily  = fetch_batch(batch, daily_period, "1d")
        weekly = fetch_batch(batch, "2y", "1wk") if need_weekly else {}
        hourly = fetch_intraday(batch, "5d", "1h")  if need_hourly   else {}
        # 15m: BullBhai needs ~1 month for 200 bars (200 * 15m = ~50h ≈ 7 trading days)
        # Use 1mo period to ensure enough bars
        if RUN_BULLBHAI:
            m15_long = fetch_intraday(batch, "1mo", "15m")
            m15      = m15_long  # reuse for DualBB 15m too
        elif RUN_DUALBB_15M:
            m15 = fetch_intraday(batch, "5d", "15m")
        else:
            m15 = {}
        h200 = fetch_intraday(batch, "1mo", "1h") if need_1h_long else {}

        for t in batch:
            d = daily.get(t)
            h = hourly.get(t)
            if d is None or len(d) < 10: continue

            # ── Existing scans ──
            if RUN_OPTION_SELL and h is not None:
                if scan_option_sell(d, h): hits["option_sell"].append(t)

            if RUN_RATHOD_BULLISH and h is not None:
                if scan_rathod_bullish(d, h): hits["rathod_bullish"].append(t)

            if RUN_NR7:
                w = weekly.get(t)
                if scan_nr7_weekly_monthly(d, w, None) if w is not None else scan_nr7(d):
                    hits["nr7"].append(t)

            if RUN_MOMENTUM:
                if scan_momentum(d): hits["momentum"].append(t)

            if RUN_DUALBB_15M or RUN_DUALBB_1H:
                try: daily_rsi_val = _f(rsi(d["Close"], 14).iloc[-1])
                except: daily_rsi_val = 50

                if RUN_DUALBB_15M:
                    m15_df = m15.get(t)
                    if m15_df is not None and len(m15_df) >= 201:
                        res = scan_dualbb_15m(m15_df, daily_rsi_val)
                        if res == "bullish": hits["dualbb_15m_bull"].append(t)
                        elif res == "bearish": hits["dualbb_15m_bear"].append(t)

                if RUN_DUALBB_1H:
                    h200_df = h200.get(t)
                    if h200_df is not None and len(h200_df) >= 201:
                        res = scan_dualbb_1h(h200_df, daily_rsi_val)
                        if res == "bullish": hits["dualbb_1h_bull"].append(t)
                        elif res == "bearish": hits["dualbb_1h_bear"].append(t)

            # ── New Chartink scans ──
            if RUN_BULLBHAI:
                m15_df = m15.get(t)
                if m15_df is not None and scan_bullbhai_momentum(d, m15_df):
                    hits["bullbhai_momentum"].append(t)

            if RUN_STB:
                if scan_short_term_breakout(d): hits["short_term_breakout"].append(t)

            if RUN_POT_BREAKOUT:
                if scan_potential_breakout(d): hits["potential_breakout"].append(t)

            if RUN_BUY_BREAKOUT:
                if scan_buy_breakout_100(d): hits["buy_breakout_100"].append(t)

            if RUN_BOTTOM_OUT:
                w = weekly.get(t)
                if scan_possible_bottom_out(d, w): hits["possible_bottom_out"].append(t)

        time.sleep(SLEEP_BETWEEN_BATCHES)

    print(f"\n{label} done")
    return hits

print("Hourly runner ready — run after 10:30 AM ET / 8:00 PM IST")


In [ ]:
# CELL 8 — RUN HOURLY SCANS
from datetime import datetime
start=datetime.now()
print(f"Started: {start.strftime('%H:%M:%S')}")
sp500_hits  = run_hourly_scans(sp500,      "S&P 500")
r2000_hits  = run_hourly_scans(r2000_only, "Russell 2000")
nifty_hits  = run_hourly_scans(nifty500,   "Nifty 500") if INCLUDE_NIFTY else {k:[] for k in HOURLY_SCAN_KEYS}
elapsed=(datetime.now()-start).seconds//60
print(f"Done in ~{elapsed} min")

names={
    "option_sell"        : "Option Sell (Chirag)",
    "rathod_bullish"     : "Rathod Bullish",
    "nr7"                : "NR7 Narrow Range Breakout",
    "momentum"           : "Momentum Scan",
    "dualbb_15m_bull"    : "Dual BB 15min Bullish",
    "dualbb_15m_bear"    : "Dual BB 15min Bearish",
    "dualbb_1h_bull"     : "Dual BB 1h Bullish",
    "dualbb_1h_bear"     : "Dual BB 1h Bearish",
    "bullbhai_momentum"  : "BullBhai Bullish Momentum",
    "short_term_breakout": "Short Term Breakout",
    "potential_breakout" : "Potential Breakout",
    "buy_breakout_100"   : "100% Buy Breakout",
    "possible_bottom_out": "Possible Bottom Out (Weekly)",
}
print()
print(f"{'='*65}")
print(f"  RESULTS  {datetime.now().strftime('%d %b %Y  %H:%M')}")
print(f"{'='*65}")
for k in HOURLY_SCAN_KEYS:
    sp=sp500_hits.get(k,[]); r2k=r2000_hits.get(k,[]); nif=nifty_hits.get(k,[])
    tot=len(sp)+len(r2k)+len(nif)
    if tot==0: print(f"  {names.get(k,k):<38}  (no hits)"); continue
    print(f"  {names.get(k,k):<38} S&P:{len(sp):>3} R2K:{len(r2k):>3} Nifty:{len(nif):>3} Total:{tot:>4}")
    if sp:  print(f"    S&P:   {', '.join(sp[:20])}")
    if r2k: print(f"    R2K:   {', '.join(r2k[:20])}")
    if nif: print(f"    Nifty: {', '.join(nif[:20])}")
print(f"{'='*65}")

# ── Save to Google Drive for Streamlit website ──────────────────
import os, pandas as pd
from datetime import datetime

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    _save_dir = '/content/drive/MyDrive/Stockbee'
    os.makedirs(_save_dir, exist_ok=True)
except Exception:
    _save_dir = '.'

_ts   = datetime.now().strftime('%Y-%m-%d %H:%M')
_rows = []
for _k in HOURLY_SCAN_KEYS:
    _label = names.get(_k, _k)
    for _t in sp500_hits.get(_k, []):  _rows.append({'Scan':_label,'Ticker':_t,'Universe':'S&P 500',   'Updated':_ts})
    for _t in r2000_hits.get(_k, []): _rows.append({'Scan':_label,'Ticker':_t,'Universe':'Russell 2000','Updated':_ts})
    for _t in nifty_hits.get(_k, []):  _rows.append({'Scan':_label,'Ticker':_t,'Universe':'Nifty 500', 'Updated':_ts})

_hourly_df = pd.DataFrame(_rows) if _rows else pd.DataFrame(columns=['Scan','Ticker','Universe','Updated'])
_csv_path  = os.path.join(_save_dir, 'hourly_hits.csv')
_hourly_df.to_csv(_csv_path, index=False)
print(f"\n✅ Saved hourly_hits.csv → {_csv_path}  ({len(_hourly_df)} rows)")
print(f"   Scans: {_hourly_df['Scan'].nunique() if not _hourly_df.empty else 0}  |  Tickers: {_hourly_df['Ticker'].nunique() if not _hourly_df.empty else 0}")


---
## Publish to Google Sheets

In [ ]:
# PUBLISH — Write results to Google Sheets
import subprocess, sys
subprocess.check_call([sys.executable,"-m","pip","install","-q",
    "gspread","gspread-dataframe","google-auth","google-auth-oauthlib"])

import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth
import pandas as pd
from datetime import datetime

# ── Auth: the correct pattern for Colab 2024+ ────────────────
# Step 1: trigger Google sign-in
auth.authenticate_user()

# Step 2: get credentials using google-auth with explicit scopes
from google.auth import default
from google.auth.transport.requests import Request

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
creds, _ = default(scopes=SCOPES)

# Step 3: force a token refresh so it's valid
try:
    creds.refresh(Request())
except Exception:
    pass  # May already be valid

# Step 4: connect gspread
gc_client = gspread.authorize(creds)
sh = gc_client.open_by_key(SHEET_ID)
ts = datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Connected: {sh.title}")
print(f"URL: https://docs.google.com/spreadsheets/d/{SHEET_ID}")

# ── Helper ────────────────────────────────────────────────────
def write_tab(df, name):
    name = name[:30]
    try:
        ws = sh.worksheet(name); ws.clear()
    except gspread.WorksheetNotFound:
        ws = sh.add_worksheet(title=name, rows=3000, cols=25)
    if not df.empty:
        set_with_dataframe(ws, df)
    print(f"  Written: {name} ({len(df)} rows)")

# ── Label maps ────────────────────────────────────────────────
all_labels = {
    "option_sell":"Option Sell (Chirag)",
    "rathod_bullish":"Rathod Intraday Bullish",
    "nr7":"NR7 Narrow Range Breakout",
    "momentum":"Momentum Scan",
    "dualbb_15m_bull":"Dual BB 15min Bullish",
    "dualbb_15m_bear":"Dual BB 15min Bearish",
    "dualbb_1h_bull":"Dual BB 1h Bullish",
    "dualbb_1h_bear"        : "Dual BB 1h Bearish",
    "bullbhai_momentum"     : "BullBhai Bullish Momentum",
    "short_term_breakout"   : "Short Term Breakout",
    "potential_breakout"    : "Potential Breakout",
    "buy_breakout_100"      : "100% Buy Breakout",
    "possible_bottom_out"   : "Possible Bottom Out (Weekly)",
}
sb_labels = {
    "scan1":"Stockbee Extreme Sales",
    "scan2":"Stockbee Sales Growth",
    "scan3":"Stockbee Growth Turnaround",
    "scan4":"Stockbee Earnings Sales",
    "momentum":"Stockbee DAMA Momentum",
}

# ── Write per-scan tabs + build summary ──────────────────────
summary = []
for k in HOURLY_SCAN_KEYS:
    label = all_labels.get(k, k)
    sp  = (sp500_hits.get(k,[]) if isinstance(sp500_hits,dict) else sp500_hits) if "sp500_hits" in dir() else []
    r2k = (r2000_hits.get(k,[]) if isinstance(r2000_hits,dict) else r2000_hits) if "r2000_hits" in dir() else []
    nif = (nifty_hits.get(k,[]) if isinstance(nifty_hits,dict) else nifty_hits) if "nifty_hits" in dir() else []
    tot = len(sp)+len(r2k)+len(nif)
    if tot == 0: continue
    summary.append({
        "Scan":label,"SP500":len(sp),"R2K":len(r2k),"Nifty":len(nif),"Total":tot,
        "SP500_tickers":", ".join(sp),
        "R2K_tickers":", ".join(r2k),
        "Nifty_tickers":", ".join(nif),
        "Updated":ts,
    })
    rows = (
        [{"Ticker":t,"Universe":"S&P 500","Updated":ts}     for t in sp]  +
        [{"Ticker":t,"Universe":"Russell 2000","Updated":ts} for t in r2k] +
        [{"Ticker":t,"Universe":"Nifty 500","Updated":ts}   for t in nif]
    )
    write_tab(pd.DataFrame(rows), label)

# ── Stockbee tabs ─────────────────────────────────────────────
if "df" in dir() and isinstance(df, pd.DataFrame):
    for k, label in sb_labels.items():
        if k in df.columns:
            sub = df[df[k]].copy()
            if not sub.empty:
                sub["Updated"] = ts
                tcol = "ticker" if "ticker" in sub.columns else sub.columns[0]
                tlist = sub[tcol].dropna().astype(str).tolist()
                summary.append({
                    "Scan":label,"SP500":"","R2K":"","Nifty":"","Total":len(sub),
                    "SP500_tickers":", ".join(tlist[:200]),
                    "R2K_tickers":"","Nifty_tickers":"","Updated":ts,
                })
                write_tab(sub, label)
    if "any_hit" in df.columns:
        hits_df = df[df["any_hit"]].copy()
        hits_df["Updated"] = ts
        write_tab(hits_df, "Stockbee All Hits")

# ── Summary tab ───────────────────────────────────────────────
if summary:
    write_tab(pd.DataFrame(summary), "Summary")

print(f"\nPublished at {ts}")
print(f"Sheet: https://docs.google.com/spreadsheets/d/{SHEET_ID}")


---


---
## 🔬 Scan Diagnostic Tool
Enter stock tickers below and run the cell. For each stock it will:
- ✅ Show PASS / ❌ FAIL for **every individual condition** of every scan
- 🟡 Flag any **NaN / missing data** in fetched OHLCV
- 📊 Print the **actual values** vs the threshold so you can compare with Chartink


In [ ]:
# @title 🔬 DIAGNOSTIC — Enter tickers to debug scan conditions
# ─────────────────────────────────────────────────────────────────────────
# EDIT THIS LIST — use .NS suffix for NSE stocks e.g. 'RELIANCE.NS'
DEBUG_TICKERS = ['RELIANCE.NS', 'INFY.NS', 'TCS.NS']  # <── change these

# Which scans to check (set False to skip)
DBG_OPTION_SELL     = True
DBG_RATHOD_BULLISH  = True
DBG_NR7             = True
DBG_MOMENTUM        = True
DBG_BULLBHAI        = True
DBG_STB             = True
DBG_POT_BREAKOUT    = True
DBG_BUY_BREAKOUT    = True
DBG_BOTTOM_OUT      = True
# ─────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

# ── Colour helpers ──
G  = '\033[92m'   # green
R  = '\033[91m'   # red
Y  = '\033[93m'   # yellow
B  = '\033[94m'   # blue
W  = '\033[97m'   # white bold
DIM= '\033[2m'    # dim
NC = '\033[0m'    # reset

def tick(v):  return f'{G}✅ PASS{NC}' if v else f'{R}❌ FAIL{NC}'
def val(v, fmt='.2f'):
    if v is None or (isinstance(v, float) and np.isnan(v)): return f'{Y}NaN{NC}'
    try:    return f'{B}{v:{fmt}}{NC}'
    except: return f'{B}{v}{NC}'

def nan_report(df, label):
    """Print NaN counts per column for a dataframe."""
    if df is None or df.empty:
        print(f'    {Y}⚠ {label}: DataFrame is None or empty{NC}')
        return
    nans = df.isnull().sum()
    nans = nans[nans > 0]
    if nans.empty:
        print(f'    {G}✅ {label}: No NaNs ({len(df)} rows){NC}')
    else:
        print(f'    {Y}⚠ {label}: NaNs found in → {dict(nans)} ({len(df)} total rows){NC}')
    # Show last 3 rows
    print(f'    {DIM}Last 3 rows:{NC}')
    with pd.option_context('display.max_columns', 10, 'display.width', 120):
        for line in df.tail(3).to_string().split('\n'):
            print(f'      {DIM}{line}{NC}')


def _f(x):
    try: return float(x)
    except: return float('nan')

def _range(d, i):
    try: return _f(d['High'].iloc[-(i+1)]) - _f(d['Low'].iloc[-(i+1)])
    except: return float('nan')


# ── Fetch data for all debug tickers ──
print(f'{W}{'='*65}{NC}')
print(f'{W}  🔬 SCAN DIAGNOSTIC  —  {len(DEBUG_TICKERS)} tickers{NC}')
print(f'{W}{'='*65}{NC}\n')

for TICKER in DEBUG_TICKERS:
    print(f'\n{W}{'─'*65}{NC}')
    print(f'{W}  📌 {TICKER}{NC}')
    print(f'{W}{'─'*65}{NC}')

    # ── Fetch all timeframes ──
    print(f'\n  {DIM}Fetching data...{NC}')
    try:
        tk = yf.Ticker(TICKER)
        d2y  = tk.history(period='2y',  interval='1d',  auto_adjust=True)
        d1y  = tk.history(period='1y',  interval='1d',  auto_adjust=True)
        wk   = tk.history(period='2y',  interval='1wk', auto_adjust=True)
        h1   = tk.history(period='5d',  interval='1h',  auto_adjust=True)
        m15  = tk.history(period='1mo', interval='15m', auto_adjust=True)
    except Exception as e:
        print(f'  {R}❌ FETCH ERROR: {e}{NC}')
        continue

    # Use 2y daily as primary
    d = d2y if not d2y.empty else d1y

    # ── Data quality report ──
    print(f'\n  {W}📊 DATA QUALITY{NC}')
    nan_report(d,   'Daily  (2y)')
    nan_report(wk,  'Weekly (2y)')
    nan_report(h1,  'Hourly (5d)')
    nan_report(m15, '15-min (1mo)')

    # ── Precompute common values ──
    try:
        close       = d['Close']
        high        = d['High']
        low         = d['Low']
        volume      = d['Volume']
        open_       = d['Open']

        last_c      = _f(close.iloc[-1])
        last_v      = _f(volume.iloc[-1])
        prev_c      = _f(close.iloc[-2])
        prev_h      = _f(high.iloc[-2])
        prev_v      = _f(volume.iloc[-2])
        two_d_high  = _f(high.iloc[-3])

        # SMAs
        s20   = _f(close.rolling(20).mean().iloc[-1])
        s40   = _f(close.rolling(40).mean().iloc[-1])
        s50   = _f(close.rolling(50).mean().iloc[-1])
        s60   = _f(close.rolling(60).mean().iloc[-1])
        v_s5  = _f(volume.rolling(5).mean().iloc[-1])
        v_s50 = _f(volume.rolling(50).mean().iloc[-1])
        v_s20 = _f(volume.rolling(20).mean().iloc[-1])

        # RSI
        def _rsi(s, n=14):
            d_ = s.diff()
            g  = d_.clip(lower=0).rolling(n).mean()
            l  = (-d_.clip(upper=0)).rolling(n).mean()
            return 100 - (100 / (1 + g / l.replace(0, np.nan)))
        rsi14     = _f(_rsi(close, 14).iloc[-1])

        today_range  = _f(high.iloc[-1]) - _f(low.iloc[-1])
        today_open   = _f(open_.iloc[-1])

        # Rolling max/min
        max5_close   = _f(close.rolling(5).max().iloc[-1])
        max120_6ago  = _f(close.rolling(120).max().iloc[-7]) if len(close) >= 127 else float('nan')
        high200      = _f(high.rolling(200).max().iloc[-1])  if len(high) >= 200 else float('nan')
        high30_now   = _f(high.rolling(30).max().iloc[-1])
        high8_30ago  = _f(high.rolling(8).max().iloc[-31])   if len(high) >= 38  else float('nan')

        # EMA for BullBhai
        ema200_15m   = _f(m15['Close'].ewm(span=200, adjust=False).mean().iloc[-1]) if m15 is not None and len(m15) >= 201 else float('nan')
        close_15m    = _f(m15['Close'].iloc[-1]) if m15 is not None and len(m15) > 0 else float('nan')

    except Exception as e:
        print(f'  {R}❌ Compute error: {e}{NC}')
        continue

    # ════════════════════════════════════════════════════════
    # SCAN: OPTION SELL
    # ════════════════════════════════════════════════════════
    if DBG_OPTION_SELL:
        print(f'\n  {W}[OPTION SELL — Chirag Rathod Bearish]{NC}')
        try:
            from pandas import Series
            if h1 is not None and len(h1) >= 4:
                tp  = (h1['High']+h1['Low']+h1['Close'])/3
                tpv = (tp * h1['Volume']).cumsum()
                cvl = h1['Volume'].cumsum()
                vwap = tpv / cvl
                b1   = h1.iloc[-2]; b2 = h1.iloc[-3]
                v1   = _f(vwap.iloc[-2])
                d_s20= _f(close.rolling(20).mean().iloc[-1])
                c1  = bool(_f(b1['Close']) < _f(b1['Open']))
                c2  = bool(_f(b1['Open'])  < _f(b1['High']))
                c3  = bool(_f(b1['Close']) > _f(b1['Low']))
                c4  = bool(_f(b1['Close']) < _f(b1['High']))
                c5  = bool(_f(b1['Close']) < v1)
                c6  = bool(_f(b1['Open'])  > v1)
                c7  = bool(_f(b1['High'])  > v1)
                c8  = bool(_f(b2['Close']) > _f(b2['Open']))
                c9  = bool(_f(b1['Open'])  > _f(b2['Open']))
                c10 = bool(_f(b1['Open'])  > _f(b2['Close']))
                c11 = bool(last_c < d_s20)
                for cond, desc, extra in [
                    (c1,  'b1 close < b1 open (bearish candle)',  f'close={val(_f(b1["Close"]))} open={val(_f(b1["Open"]))}'),
                    (c2,  'b1 open < b1 high',                    f'open={val(_f(b1["Open"]))} high={val(_f(b1["High"]))}'),
                    (c3,  'b1 close > b1 low',                    f''),
                    (c4,  'b1 close < b1 high',                   f''),
                    (c5,  'b1 close < VWAP',                      f'close={val(_f(b1["Close"]))} vwap={val(v1)}'),
                    (c6,  'b1 open > VWAP',                       f'open={val(_f(b1["Open"]))} vwap={val(v1)}'),
                    (c7,  'b1 high > VWAP',                       f''),
                    (c8,  'b2 close > b2 open (prior bullish)',    f''),
                    (c9,  'b1 open > b2 open',                    f''),
                    (c10, 'b1 open > b2 close',                   f''),
                    (c11, 'daily close < SMA20',                  f'close={val(last_c)} sma20={val(d_s20)}'),
                ]:
                    print(f'    {tick(cond)}  {desc}  {DIM}{extra}{NC}')
            else:
                print(f'    {Y}⚠ Insufficient 1h data (need 4 bars, have {len(h1) if h1 is not None else 0}){NC}')
        except Exception as e:
            print(f'    {R}Error: {e}{NC}')

    # ════════════════════════════════════════════════════════
    # SCAN: NR7
    # ════════════════════════════════════════════════════════
    if DBG_NR7:
        print(f'\n  {W}[NR7 — Narrow Range 7 / 100% Buy Breakout]{NC}')
        try:
            r0 = today_range
            ranges = [(_range(d, i), i) for i in range(1, 8)]
            print(f'    Today range: {val(r0)}')
            for ri, i in ranges:
                ok = bool(r0 < ri)
                print(f'    {tick(ok)}  today range < {i}d ago range  {DIM}({val(r0)} < {val(ri)}){NC}')
            c_co = bool(last_c > today_open)
            c_cp = bool(last_c > prev_c)
            c_pv = bool(prev_v > 10000)
            c_vs = bool(last_v > prev_v * 1.25)
            c_sm = bool(s20 > s40 > s60)
            c_wk = bool(_f(d['Close'].iloc[-1]) > _f(d['Open'].iloc[-5]))
            c_mo = bool(_f(d['Close'].iloc[-1]) > _f(d['Open'].iloc[-22]))
            for cond, desc, extra in [
                (c_co, 'close > open (bullish candle)',      f'close={val(last_c)} open={val(today_open)}'),
                (c_cp, 'close > prior close',               f'close={val(last_c)} prev={val(prev_c)}'),
                (c_pv, 'prior volume > 10,000',             f'prev_vol={val(prev_v,".0f")}'),
                (c_vs, 'volume > prior_vol × 1.25',         f'today={val(last_v,".0f")} prev={val(prev_v,".0f")} threshold={val(prev_v*1.25,".0f")}'),
                (c_sm, 'SMA20 > SMA40 > SMA60',            f'sma20={val(s20)} sma40={val(s40)} sma60={val(s60)}'),
                (c_wk, 'weekly approx: close > 5d-ago open',f'close={val(last_c)} 5d_open={val(_f(d["Open"].iloc[-5]))}'),
                (c_mo, 'monthly approx: close > 22d-ago open',f'close={val(last_c)} 22d_open={val(_f(d["Open"].iloc[-22]))}'),
            ]:
                print(f'    {tick(cond)}  {desc}  {DIM}{extra}{NC}')
        except Exception as e:
            print(f'    {R}Error: {e}{NC}')

    # ════════════════════════════════════════════════════════
    # SCAN: MOMENTUM
    # ════════════════════════════════════════════════════════
    if DBG_MOMENTUM:
        print(f'\n  {W}[MOMENTUM SCAN]{NC}')
        try:
            conds = [
                (bool(last_c > 50),         'close > 50',             f'close={val(last_c)}'),
                (bool(last_v > v_s20),      'volume > SMA(vol,20)',   f'vol={val(last_v,".0f")} sma20vol={val(v_s20,".0f")}'),
                (bool(s20 > s50),           'SMA20 > SMA50',          f'sma20={val(s20)} sma50={val(s50)}'),
                (bool(rsi14 > 50),          'RSI(14) > 50',           f'rsi={val(rsi14)}'),
                (bool(last_c > s50),        'close > SMA50',          f'close={val(last_c)} sma50={val(s50)}'),
                (bool(last_c > prev_h),     'close > yesterday high', f'close={val(last_c)} prev_high={val(prev_h)}'),
                (bool(v_s20 > 500_000),     'avg vol 20d > 500K',     f'avg_vol={val(v_s20,".0f")}'),
            ]
            for cond, desc, extra in conds:
                print(f'    {tick(cond)}  {desc}  {DIM}{extra}{NC}')
        except Exception as e:
            print(f'    {R}Error: {e}{NC}')

    # ════════════════════════════════════════════════════════
    # SCAN: BULLBHAI MOMENTUM
    # ════════════════════════════════════════════════════════
    if DBG_BULLBHAI:
        print(f'\n  {W}[BULLBHAI BULLISH MOMENTUM]{NC}')
        try:
            bars_15m = len(m15) if m15 is not None else 0
            c1 = bool(close_15m > ema200_15m) if not np.isnan(ema200_15m) else False
            c2 = bool(last_c > two_d_high)
            c3 = bool(rsi14 > 51)
            print(f'    {DIM}15m bars available: {bars_15m} (need 201){NC}')
            for cond, desc, extra in [
                (c1, '15m close > EMA(200) on 15m',    f'15m_close={val(close_15m)} ema200_15m={val(ema200_15m)}'),
                (c2, 'daily close > 2 days ago high',  f'close={val(last_c)} 2d_high={val(two_d_high)}'),
                (c3, 'daily RSI(14) > 51',             f'rsi={val(rsi14)}'),
            ]:
                print(f'    {tick(cond)}  {desc}  {DIM}{extra}{NC}')
        except Exception as e:
            print(f'    {R}Error: {e}{NC}')

    # ════════════════════════════════════════════════════════
    # SCAN: SHORT TERM BREAKOUT
    # ════════════════════════════════════════════════════════
    if DBG_STB:
        print(f'\n  {W}[SHORT TERM BREAKOUT]{NC}')
        try:
            c1 = bool(max5_close > max120_6ago * 1.05) if not np.isnan(max120_6ago) else False
            c2 = bool(last_v > v_s5)
            c3 = bool(last_c > prev_c)
            print(f'    {DIM}Daily bars: {len(d)} (need 130){NC}')
            for cond, desc, extra in [
                (c1, '5d max close > 120d max close (6d ago) × 1.05',
                     f'max5={val(max5_close)} max120_6ago={val(max120_6ago)} threshold={val(max120_6ago*1.05 if not np.isnan(max120_6ago) else float("nan"))}'),
                (c2, 'volume > SMA(vol,5)',  f'vol={val(last_v,".0f")} sma5vol={val(v_s5,".0f")}'),
                (c3, 'close > prior close',  f'close={val(last_c)} prev={val(prev_c)}'),
            ]:
                print(f'    {tick(cond)}  {desc}  {DIM}{extra}{NC}')
        except Exception as e:
            print(f'    {R}Error: {e}{NC}')

    # ════════════════════════════════════════════════════════
    # SCAN: POTENTIAL BREAKOUT
    # ════════════════════════════════════════════════════════
    if DBG_POT_BREAKOUT:
        print(f'\n  {W}[POTENTIAL BREAKOUT]{NC}')
        try:
            c1 = bool(last_c * 1.05 > high200) if not np.isnan(high200) else False
            c2 = bool(high30_now <= high8_30ago) if not np.isnan(high8_30ago) else False
            c3 = bool(last_v > v_s50)
            c4 = bool(last_c > 90)
            print(f'    {DIM}Daily bars: {len(d)} (need 230){NC}')
            for cond, desc, extra in [
                (c1, 'close × 1.05 > 200d high (within 5% of yearly high)',
                     f'close×1.05={val(last_c*1.05)} 200d_high={val(high200)}'),
                (c2, '30d high <= 8d high from 30d ago (contracting range)',
                     f'high30={val(high30_now)} high8_30ago={val(high8_30ago)}'),
                (c3, 'volume > SMA(vol,50)', f'vol={val(last_v,".0f")} sma50vol={val(v_s50,".0f")}'),
                (c4, 'close > 90',           f'close={val(last_c)}'),
            ]:
                print(f'    {tick(cond)}  {desc}  {DIM}{extra}{NC}')
        except Exception as e:
            print(f'    {R}Error: {e}{NC}')

    # ════════════════════════════════════════════════════════
    # SCAN: 100% BUY BREAKOUT
    # ════════════════════════════════════════════════════════
    if DBG_BUY_BREAKOUT:
        print(f'\n  {W}[100% BUY BREAKOUT]{NC}')
        try:
            range_checks = [(bool(today_range > _range(d, i)), i) for i in range(1, 8)]
            all_ranges_ok = all(c for c, _ in range_checks)
            print(f'    Today range: {val(today_range)}')
            for ok, i in range_checks:
                print(f'    {tick(ok)}  today range > {i}d ago  {DIM}({val(today_range)} > {val(_range(d,i))}){NC}')
            for cond, desc, extra in [
                (bool(last_c > today_open),        'close > open',              f'close={val(last_c)} open={val(today_open)}'),
                (bool(last_c > prev_c),            'close > prior close',       f'close={val(last_c)} prev={val(prev_c)}'),
                (bool(prev_v > 10000),             'prior volume > 10,000',     f'prev_vol={val(prev_v,".0f")}'),
                (bool(last_v > prev_v * 1.25),     'volume > prior × 1.25',    f'today={val(last_v,".0f")} threshold={val(prev_v*1.25,".0f")}'),
                (bool(s20 > s40 > s60),            'SMA20 > SMA40 > SMA60',    f'sma20={val(s20)} sma40={val(s40)} sma60={val(s60)}'),
                (bool(last_c > _f(d["Open"].iloc[-5])),  'weekly: close > 5d open',  f'close={val(last_c)} 5d_open={val(_f(d["Open"].iloc[-5]))}'),
                (bool(last_c > _f(d["Open"].iloc[-22])), 'monthly: close > 22d open', f'close={val(last_c)} 22d_open={val(_f(d["Open"].iloc[-22]))}'),
            ]:
                print(f'    {tick(cond)}  {desc}  {DIM}{extra}{NC}')
        except Exception as e:
            print(f'    {R}Error: {e}{NC}')

    # ════════════════════════════════════════════════════════
    # SCAN: POSSIBLE BOTTOM OUT WEEKLY
    # ════════════════════════════════════════════════════════
    if DBG_BOTTOM_OUT:
        print(f'\n  {W}[POSSIBLE BOTTOM OUT — WEEKLY]{NC}')
        try:
            if wk is None or len(wk) < 6:
                print(f'    {Y}⚠ Insufficient weekly data (have {len(wk) if wk is not None else 0}, need 6){NC}')
            else:
                def wh(i): return _f(wk['High'].iloc[i])
                def wl(i): return _f(wk['Low'].iloc[i])
                def wc(i): return _f(wk['Close'].iloc[i])
                def wo(i): return _f(wk['Open'].iloc[i])
                print(f'    {DIM}Weekly bars: {len(wk)}{NC}')
                print(f'    {DIM}Week data (H/L/O/C):{NC}')
                for lbl, i in [('4w ago',-5),('3w ago',-4),('2w ago',-3),('1w ago',-2),('This wk',-1)]:
                    print(f'      {DIM}{lbl}: H={wh(i):.2f} L={wl(i):.2f} O={wo(i):.2f} C={wc(i):.2f}{NC}')
                conds_wk = [
                    (bool(wh(-5)>wh(-4) and wl(-5)>wl(-4) and wc(-5)<wo(-5)),
                     '4w ago: higher H&L than 3w ago + bearish candle',
                     f'4w H={val(wh(-5))} > 3w H={val(wh(-4))}  4w L={val(wl(-5))} > 3w L={val(wl(-4))}  4w bearish={val(wc(-5)<wo(-5))}'),
                    (bool(wh(-4)>wh(-3) and wl(-4)>wl(-3) and wc(-4)<wo(-4)),
                     '3w ago: higher H&L than 2w ago + bearish candle', ''),
                    (bool(wh(-3)>wh(-2) and wl(-3)>wl(-2) and wc(-3)<wo(-3)),
                     '2w ago: higher H&L than 1w ago + bearish candle', ''),
                    (bool(wh(-1)>wh(-2) and wl(-1)>wl(-2) and wc(-1)>wo(-1)),
                     'This week: higher H&L + bullish reversal (close>open)',
                     f'H={val(wh(-1))} > prev H={val(wh(-2))}  L={val(wl(-1))} > prev L={val(wl(-2))}'),
                    (bool(wc(-1) >= 100),
                     'Weekly close >= 100', f'close={val(wc(-1))}'),
                ]
                for cond, desc, extra in conds_wk:
                    print(f'    {tick(cond)}  {desc}  {DIM}{extra}{NC}')
        except Exception as e:
            print(f'    {R}Error: {e}{NC}')

    # ════════════════════════════════════════════════════════
    # SCAN: RATHOD BULLISH
    # ════════════════════════════════════════════════════════
    if DBG_RATHOD_BULLISH:
        print(f'\n  {W}[RATHOD INTRADAY BULLISH]{NC}')
        try:
            if h1 is None or len(h1) < 8:
                print(f'    {Y}⚠ Insufficient 1h data{NC}')
            else:
                h_s20 = _f(h1['Close'].rolling(20).mean().iloc[-1])
                h_c   = _f(h1['Close'].iloc[-1])
                h_rsi = _f(_rsi(h1['Close'], 14).iloc[-1])
                conds_r = [
                    (bool(last_c > s20),      'daily close > daily SMA20',  f'close={val(last_c)} sma20={val(s20)}'),
                    (bool(rsi14 > 40),         'daily RSI(14) > 40',         f'rsi={val(rsi14)}'),
                    (bool(h_c > h_s20),        '1h close > 1h SMA20',        f'h_close={val(h_c)} h_sma20={val(h_s20)}'),
                    (bool(h_rsi > 40),         '1h RSI(14) > 40',            f'h_rsi={val(h_rsi)}'),
                ]
                for cond, desc, extra in conds_r:
                    print(f'    {tick(cond)}  {desc}  {DIM}{extra}{NC}')
        except Exception as e:
            print(f'    {R}Error: {e}{NC}')

print(f'\n{W}{'='*65}{NC}')
print(f'{W}  🔬 Diagnostic complete. Compare ❌ FAIL rows with Chartink.{NC}')
print(f'{W}  If values show {Y}NaN{W} — that data is missing from Yahoo Finance.{NC}')
print(f'{W}{'='*65}{NC}')


In [ ]:
# ── SAVE TO GOOGLE DRIVE folder: scan_result ──────────────────────────
# Folder: https://drive.google.com/drive/folders/1qFAeut_82HvPP2-tntvxmxVl-uysms9c
# This cell runs independently — no drive.mount needed
import os, pandas as pd
from datetime import datetime
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaInMemoryUpload

auth.authenticate_user()
from google.auth import default
_creds, _ = default()
_svc = build('drive', 'v3', credentials=_creds)
_FOLDER_ID = "1qFAeut_82HvPP2-tntvxmxVl-uysms9c"

def upload_csv(df, filename):
    csv_bytes = df.to_csv(index=False).encode('utf-8')
    media = MediaInMemoryUpload(csv_bytes, mimetype='text/csv', resumable=False)
    q = "name='" + filename + "' and '" + _FOLDER_ID + "' in parents and trashed=false"
    existing = _svc.files().list(q=q, fields='files(id,name)').execute().get('files', [])
    if existing:
        _svc.files().update(fileId=existing[0]['id'], media_body=media).execute()
        print("  Updated:", filename, "->", len(df), "rows ->", "https://drive.google.com/drive/folders/1qFAeut_82HvPP2-tntvxmxVl-uysms9c")
    else:
        meta = dict(name=filename, parents=[_FOLDER_ID])
        _svc.files().create(body=meta, media_body=media, fields='id').execute()
        print("  Created:", filename, "->", len(df), "rows ->", "https://drive.google.com/drive/folders/1qFAeut_82HvPP2-tntvxmxVl-uysms9c")

_ts = datetime.now().strftime('%Y-%m-%d %H:%M')
_rows = []
for _k in HOURLY_SCAN_KEYS:
    _lbl = names.get(_k, _k)
    for _t in sp500_hits.get(_k, []):  _rows.append({'Scan':_lbl,'Ticker':_t,'Universe':'S&P 500',    'Updated':_ts})
    for _t in r2000_hits.get(_k, []):  _rows.append({'Scan':_lbl,'Ticker':_t,'Universe':'Russell 2000','Updated':_ts})
    for _t in nifty_hits.get(_k, []):  _rows.append({'Scan':_lbl,'Ticker':_t,'Universe':'Nifty 500',  'Updated':_ts})
_df = pd.DataFrame(_rows) if _rows else pd.DataFrame(columns=['Scan','Ticker','Universe','Updated'])
upload_csv(_df, 'hourly_hits.csv')
